# Week 4 - Research Agent

This notebook implements the Week 4 assignment: a research agent with web search, session memory, tool-call logging, and file reading, combined into a single agent that can answer multi-hop questions.

The agent runs on OpenRouter using OpenAI-compatible tool calling, and uses SerpApi for live web search. It follows an Agent, Tools, Memory, LLM architecture: the agent orchestrates each turn, tools perform actions, memory persists state across the session, and the LLM decides which tool to call and when.

The Demonstration section shows the agent using all four tools together within a single session to answer a multi-hop question, which is the core requirement of this assignment.

## 1. Installation

Installs the packages used in this notebook: `openai` for the OpenRouter client, `requests` for SerpApi calls, `pypdf` for PDF text extraction, `fpdf2` to generate sample files for the demonstration, and `python-dotenv` to load API keys from a local `.env` file.

Package imports are placed with each component, immediately before first use, rather than gathered into a single import cell, so each component's dependencies stay visible where they are introduced.

In [1]:
%pip install -q openai requests pypdf fpdf2 python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

Loads `OPENROUTER_API_KEY` and `SERPAPI_API_KEY`, first from a local `.env` file and then, if not found, via an interactive prompt. `MODEL_NAME` selects the OpenRouter model; `openrouter/free` auto-selects a free model that supports tool calling.

In [2]:
import os
from getpass import getpass

from dotenv import load_dotenv

# Loads key=value pairs from a local .env file (if one exists next to this
# notebook) into the process environment. Safe to call with no .env present.
load_dotenv()


def _require_secret(env_var: str, prompt: str) -> str:
    """Fetch a required secret from the environment, prompting once if missing.

    Raises immediately with a clear message if the value is blank, so a
    missing/empty key fails at setup time instead of surfacing later as a
    confusing 401 from deep inside a tool call.
    """
    value = os.environ.get(env_var) or getpass(prompt)
    if not value or not value.strip():
        raise ValueError(f"{env_var} is required but was left blank.")
    os.environ[env_var] = value
    return value


OPENROUTER_API_KEY = _require_secret("OPENROUTER_API_KEY", "Enter your OpenRouter API key: ")
SERPAPI_API_KEY = _require_secret("SERPAPI_API_KEY", "Enter your SerpApi API key: ")

# "openrouter/free" auto-selects a free model that supports tool calling, so this
# notebook keeps working even as the free-model lineup rotates. For fully
# deterministic behavior, pin a specific free model instead, e.g.:
#   MODEL_NAME = "openai/gpt-oss-20b:free"
#   MODEL_NAME = "qwen/qwen3-coder:free"
MODEL_NAME = "openrouter/free"

## 3. Logger Hook

`ToolCallLogger` wraps every tool function and records its inputs, start and end timestamps, duration, and outcome. This is the logging hook required by the assignment: every tool call is captured automatically, without the tool implementations needing to know logging exists.

In [3]:
import functools
import inspect
import json
import time
from datetime import datetime
from typing import Any, Callable, Dict, List, Optional


class ToolCallLogger:
    """Hook that records every tool invocation with timing metadata.

    Wrap any tool function with `.wrap` (aliased below as `log_tool_call`) to
    get automatic logging of inputs, timestamps, duration, and success/failure
    — without that function needing to know logging exists.
    """

    def __init__(self, log_path: str = "tool_calls.log") -> None:
        self.log_path = log_path
        self.records: List[Dict[str, Any]] = []

    def _persist(self, record: Dict[str, Any]) -> None:
        with open(self.log_path, "a") as f:
            f.write(json.dumps(record) + "\n")

    def _bind_inputs(self, func: Callable, args: tuple, kwargs: dict) -> Dict[str, Any]:
        """Resolve positional + keyword call args to a {param_name: value} dict."""
        try:
            bound = inspect.signature(func).bind(*args, **kwargs)
            bound.apply_defaults()
            return dict(bound.arguments)
        except TypeError:
            # Fallback for the rare case a call doesn't match the signature.
            return {"args": args, "kwargs": kwargs}

    def wrap(self, func: Callable) -> Callable:
        """Decorator: logs a tool call's inputs, timing, and outcome."""

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            call_id = len(self.records) + 1
            call_inputs = self._bind_inputs(func, args, kwargs)
            start_perf = time.perf_counter()
            record: Dict[str, Any] = {
                "call_id": call_id,
                "tool": func.__name__,
                "inputs": call_inputs,
                "start_time": datetime.now().isoformat(timespec="milliseconds"),
            }
            try:
                result = func(*args, **kwargs)
                record.update(self._outcome_fields(start_perf, status="success", result=result))
                self.records.append(record)
                self._persist(record)
                print(f"[HOOK] #{call_id} {func.__name__}({call_inputs}) -> {record['duration_ms']}ms")
                return result
            except Exception as exc:
                record.update(self._outcome_fields(start_perf, status="error", error=exc))
                self.records.append(record)
                self._persist(record)
                print(f"[HOOK] #{call_id} {func.__name__} FAILED: {exc}")
                raise

        return wrapper

    @staticmethod
    def _outcome_fields(
        start_perf: float,
        status: str,
        result: Optional[Any] = None,
        error: Optional[Exception] = None,
    ) -> Dict[str, Any]:
        fields: Dict[str, Any] = {
            "end_time": datetime.now().isoformat(timespec="milliseconds"),
            "duration_ms": round((time.perf_counter() - start_perf) * 1000, 2),
            "status": status,
        }
        if status == "success":
            preview = str(result)
            fields["result_preview"] = preview[:200] + ("..." if len(preview) > 200 else "")
        else:
            fields["error"] = str(error)
        return fields


tool_logger = ToolCallLogger()
log_tool_call = tool_logger.wrap  # keeps `@log_tool_call` usable exactly as before
TOOL_CALL_LOG = tool_logger.records  # backward-compatible alias used later in the notebook

## 4. Memory

`AgentMemory` stores the full conversation history sent to the LLM on each turn, plus an explicit key-value fact store the agent can write to and read from during a session. This is what lets the agent recall facts it learned earlier in the same conversation.

In [4]:
class AgentMemory:
    """Session memory for the agent: conversation history + an explicit fact store."""

    def __init__(self) -> None:
        self.conversation: List[Dict[str, Any]] = []
        self.facts: Dict[str, str] = {}

    def add_user_message(self, content: str) -> None:
        """Append a plain user turn to the conversation history."""
        self.conversation.append({"role": "user", "content": content})

    def add_message(self, message: Dict[str, Any]) -> None:
        """Append a raw OpenAI-format message (assistant tool-call turns, tool
        results). Kept separate from `add_user_message` because those messages
        carry extra keys (`tool_calls`, `tool_call_id`) a plain user turn never needs.
        """
        self.conversation.append(message)

    def remember(self, key: str, value: str) -> str:
        """Store a fact under a normalized key. Returns a human-readable confirmation."""
        normalized_key = key.strip().lower()
        self.facts[normalized_key] = value
        return f"Remembered '{key}' = '{value}'"

    def recall(self, key: str) -> str:
        """Look up a fact by key, falling back to a substring match across
        stored keys so a close-but-not-exact key (e.g. 'hq city' vs
        'acme_hq_city') still resolves instead of returning a hard miss.
        """
        normalized_key = key.strip().lower()
        if normalized_key in self.facts:
            return self.facts[normalized_key]
        fuzzy_matches = {
            k: v for k, v in self.facts.items() if normalized_key in k or k in normalized_key
        }
        if fuzzy_matches:
            return json.dumps(fuzzy_matches)
        return "No fact found for that key."

    def facts_summary(self) -> str:
        """Render all stored facts as a bullet list for injection into the system prompt."""
        if not self.facts:
            return "No facts stored yet."
        return "\n".join(f"- {k}: {v}" for k, v in self.facts.items())


memory = AgentMemory()

## 5. Web Search Tool

`web_search` queries SerpApi's Google Search API and returns the top organic results. It is one of four tools available to the agent, and returns a structured result (`status`, `results`, or `error`) so the LLM can distinguish a successful empty search from a failed request.

In [5]:
import requests

SERPAPI_ENDPOINT = "https://serpapi.com/search"


@log_tool_call
def web_search(query: str, count: int = 5) -> Dict[str, Any]:
    """Search the live web via SerpApi's Google Search API.

    Returns a structured dict, always with a "status" key:
      {"status": "success", "query": ..., "results": [{"title","url","snippet"}, ...]}
      {"status": "success", "query": ..., "results": [], "message": "..."}   # no hits
      {"status": "error",   "query": ..., "error": "<human-readable reason>"}
    """
    if not SERPAPI_API_KEY:
        return {"status": "error", "query": query, "error": "SERPAPI_API_KEY is not configured."}

    count = max(1, min(count, 10))  # guard against unreasonable values from the model
    params = {
        "engine": "google",
        "q": query,
        "num": count,
        "api_key": SERPAPI_API_KEY,
    }

    try:
        response = requests.get(SERPAPI_ENDPOINT, params=params, timeout=15)
        response.raise_for_status()
    except requests.exceptions.Timeout:
        return {"status": "error", "query": query, "error": "SerpApi request timed out."}
    except requests.exceptions.HTTPError as exc:
        status_code = exc.response.status_code if exc.response is not None else "unknown"
        return {"status": "error", "query": query, "error": f"SerpApi returned HTTP {status_code}."}
    except requests.exceptions.RequestException as exc:
        return {"status": "error", "query": query, "error": f"SerpApi request failed: {exc}"}

    try:
        data = response.json()
    except ValueError:
        return {"status": "error", "query": query, "error": "SerpApi returned invalid JSON."}

    # SerpApi can return HTTP 200 with an "error" field instead of a non-2xx
    # status (e.g. invalid api_key, exhausted plan) -- check for that explicitly
    # rather than assuming a 200 response always means success.
    if "error" in data:
        return {"status": "error", "query": query, "error": data["error"]}

    raw_results = data.get("organic_results", [])[:count]
    results = [
        {"title": item.get("title"), "url": item.get("link"), "snippet": item.get("snippet")}
        for item in raw_results
    ]

    if not results:
        return {"status": "success", "query": query, "results": [], "message": "No results found for this query."}

    return {"status": "success", "query": query, "results": results}


## 6. File Reader Tool

`read_file` reads the text content of a local `.txt` or `.pdf` file, using `pypdf` for PDF extraction. It returns the same structured result shape as the other tools, with explicit handling for missing files, unsupported extensions, and unreadable PDFs.

In [6]:
from pypdf import PdfReader
from pypdf.errors import PyPdfError


@log_tool_call
def read_file(path: str, max_chars: int = 5000) -> Dict[str, Any]:
    """Read a local .txt or .pdf file and return its text content (truncated).

    Returns a structured dict, always with a "status" key:
      {"status": "success", "path": ..., "content": "...", "truncated": bool}
      {"status": "error",   "path": ..., "error": "<human-readable reason>"}
    """
    if not os.path.exists(path):
        return {"status": "error", "path": path, "error": f"File not found: '{path}'"}

    extension = os.path.splitext(path)[1].lower()

    try:
        if extension == ".txt":
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()
        elif extension == ".pdf":
            reader = PdfReader(path)
            if not reader.pages:
                return {"status": "error", "path": path, "error": "PDF has no readable pages."}
            text = "\n".join(page.extract_text() or "" for page in reader.pages)
        else:
            return {
                "status": "error",
                "path": path,
                "error": f"Unsupported file type '{extension}'. Only .txt and .pdf are supported.",
            }
    except (OSError, UnicodeDecodeError) as exc:
        return {"status": "error", "path": path, "error": f"Could not read text file: {exc}"}
    except PyPdfError as exc:
        return {"status": "error", "path": path, "error": f"Could not parse PDF: {exc}"}

    if not text.strip():
        return {"status": "error", "path": path, "error": "File was read but contained no extractable text."}

    truncated = len(text) > max_chars
    content = text[:max_chars] + ("..." if truncated else "")
    return {"status": "success", "path": path, "content": content, "truncated": truncated}

## 7. Memory Tools

`remember_fact` and `recall_fact` expose `AgentMemory`'s fact store to the LLM as tools. This lets the agent decide, mid-conversation, when to save or retrieve a fact, rather than relying on facts being re-read implicitly from the transcript.

In [7]:
@log_tool_call
def remember_fact(key: str, value: str) -> Dict[str, Any]:
    """Persist a fact to session memory as a normalized key/value pair."""
    if not key or not key.strip():
        return {"status": "error", "error": "Cannot remember a fact with an empty key."}
    confirmation = memory.remember(key, value)
    return {"status": "success", "key": key, "value": value, "message": confirmation}


@log_tool_call
def recall_fact(key: str) -> Dict[str, Any]:
    """Retrieve a previously remembered fact from session memory by key."""
    if not key or not key.strip():
        return {"status": "error", "error": "Cannot recall a fact with an empty key."}
    result = memory.recall(key)
    if result == "No fact found for that key.":
        return {"status": "not_found", "key": key, "message": result}
    return {"status": "success", "key": key, "value": result}

### Tool Validation

A short, deterministic check that `read_file` and `recall_fact` handle invalid input correctly (missing file, unsupported extension, empty key, unknown fact). It runs without any network or LLM calls.

In [8]:
with open("scratch_unsupported.md", "w") as f:
    f.write("A scratch file with an extension read_file() intentionally rejects.")

print("Missing file       ->", read_file("this_file_does_not_exist.txt"))
print("Unsupported ext     ->", read_file("scratch_unsupported.md"))
print("Empty fact key       ->", recall_fact("   "))
print("Recall before remember ->", recall_fact("a_fact_nobody_stored_yet"))

[HOOK] #1 read_file({'path': 'this_file_does_not_exist.txt', 'max_chars': 5000}) -> 0.07ms
Missing file       -> {'status': 'error', 'path': 'this_file_does_not_exist.txt', 'error': "File not found: 'this_file_does_not_exist.txt'"}
[HOOK] #2 read_file({'path': 'scratch_unsupported.md', 'max_chars': 5000}) -> 0.28ms
Unsupported ext     -> {'status': 'error', 'path': 'scratch_unsupported.md', 'error': "Unsupported file type '.md'. Only .txt and .pdf are supported."}
[HOOK] #3 recall_fact({'key': '   '}) -> 0.02ms
Empty fact key       -> {'status': 'error', 'error': 'Cannot recall a fact with an empty key.'}
[HOOK] #4 recall_fact({'key': 'a_fact_nobody_stored_yet'}) -> 0.05ms
Recall before remember -> {'status': 'not_found', 'key': 'a_fact_nobody_stored_yet', 'message': 'No fact found for that key.'}


## 8. Tool Schemas

Defines the OpenAI-compatible function-calling schema for each tool (`web_search`, `read_file`, `remember_fact`, `recall_fact`) and maps tool names to their Python implementations. OpenRouter uses these schemas to decide which tool to call and with what arguments.

In [9]:
def build_tool_schema(
    name: str, description: str, properties: Dict[str, Any], required: List[str]
) -> Dict[str, Any]:
    """Build one OpenAI/OpenRouter-format function-calling tool schema entry."""
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {"type": "object", "properties": properties, "required": required},
        },
    }


TOOLS_SCHEMA = [
    build_tool_schema(
        name="web_search",
        description=(
            "Search the live web for current information using SerpApi's Google Search API. "
            "Use this for facts that may be recent or that you don't already know."
        ),
        properties={
            "query": {"type": "string", "description": "The search query."},
            "count": {"type": "integer", "description": "Number of results to return (1-10).", "default": 5},
        },
        required=["query"],
    ),
    build_tool_schema(
        name="read_file",
        description="Read the text content of a local .txt or .pdf file.",
        properties={"path": {"type": "string", "description": "Path to the .txt or .pdf file."}},
        required=["path"],
    ),
    build_tool_schema(
        name="remember_fact",
        description=(
            "Save a specific fact to long-term session memory as a key/value pair "
            "so it can be recalled later in the conversation."
        ),
        properties={"key": {"type": "string"}, "value": {"type": "string"}},
        required=["key", "value"],
    ),
    build_tool_schema(
        name="recall_fact",
        description="Retrieve a previously remembered fact from session memory by key.",
        properties={"key": {"type": "string"}},
        required=["key"],
    ),
]

TOOL_FUNCTIONS: Dict[str, Callable[..., Dict[str, Any]]] = {
    "web_search": web_search,
    "read_file": read_file,
    "remember_fact": remember_fact,
    "recall_fact": recall_fact,
}

## 9. Research Agent

`ResearchAgent` implements the Agent, Tools, Memory, LLM loop: it builds the message list from memory, calls OpenRouter, executes any requested tool calls through the logging hook, and repeats until the model returns a final answer. This component ties the other sections together into a single working agent.

In [10]:
from openai import OpenAI, OpenAIError

# OpenRouter is OpenAI-compatible: same SDK, different base_url + key.
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)

# Optional but recommended by OpenRouter for analytics/rankings on their leaderboard.
EXTRA_HEADERS = {"HTTP-Referer": "https://localhost", "X-Title": "Week4 Research Agent"}

MAX_LLM_RETRIES = 3
RETRY_BACKOFF_SECONDS = 1.5

SYSTEM_PROMPT = """You are a research agent with four tools: web_search, read_file,
remember_fact, and recall_fact.

Every tool returns a JSON object with a "status" field ("success", "error", or
"not_found" for recall_fact). Always check "status" before using a result:
- On "error", do not invent an answer. Try a different approach (rephrase the
  search query, double-check the file path) or tell the user what went wrong.
- On "not_found" from recall_fact, the fact was never stored — go get it another
  way (search or file) instead of guessing.

For multi-hop questions, break the problem into steps: gather each piece of
information with a tool call, save important intermediate facts with
remember_fact, and reuse them with recall_fact instead of re-deriving them.

Ground every claim in a tool result or a remembered fact rather than relying on
your own background knowledge when a tool could verify it. Mention where each
fact came from (web search or file) in your final answer. Be concise."""


class ResearchAgent:
    """Orchestrates the Agent -> Tools -> Memory -> LLM loop.

    This class owns *turn-taking only*. It has no tool-specific logic (that
    lives in the tool functions) and no LLM-provider-specific logic beyond the
    single `_call_llm` method, which is the only place that would need to
    change if the backend were swapped again.
    """

    def __init__(self, memory: AgentMemory, model: str = MODEL_NAME, max_turns: int = 8) -> None:
        self.memory = memory
        self.model = model
        self.max_turns = max_turns

    def _build_messages(self) -> List[Dict[str, Any]]:
        """Compose system prompt (with current memory facts injected) + conversation history."""
        system_message = {
            "role": "system",
            "content": f"{SYSTEM_PROMPT}\n\nFacts currently stored in memory:\n{self.memory.facts_summary()}",
        }
        return [system_message] + self.memory.conversation

    def _call_llm(self):
        """Call OpenRouter, retrying on transient failures (rate limits, dropped
        connections, timeouts) instead of letting one hiccup end the session."""
        last_error: Optional[Exception] = None
        for attempt in range(1, MAX_LLM_RETRIES + 1):
            try:
                return client.chat.completions.create(
                    model=self.model,
                    messages=self._build_messages(),
                    tools=TOOLS_SCHEMA,
                    max_tokens=1024,
                    extra_headers=EXTRA_HEADERS,
                )
            except OpenAIError as exc:
                last_error = exc
                print(f"[AGENT] OpenRouter call failed (attempt {attempt}/{MAX_LLM_RETRIES}): {exc}")
                if attempt < MAX_LLM_RETRIES:
                    time.sleep(RETRY_BACKOFF_SECONDS * attempt)
        raise RuntimeError(f"OpenRouter request failed after {MAX_LLM_RETRIES} attempts: {last_error}")

    def _execute_tool_call(self, tool_call) -> Dict[str, str]:
        """Run one requested tool call (via the logging hook) and package the
        result as an OpenAI-format tool message."""
        tool_name = tool_call.function.name
        tool_fn = TOOL_FUNCTIONS.get(tool_name)

        try:
            tool_args = json.loads(tool_call.function.arguments or "{}")
        except json.JSONDecodeError:
            output: Any = {"status": "error", "error": f"Model sent invalid JSON arguments for '{tool_name}'."}
        else:
            if tool_fn is None:
                output = {"status": "error", "error": f"Unknown tool requested: '{tool_name}'."}
            else:
                try:
                    output = tool_fn(**tool_args)
                except Exception as exc:  # tools handle their own errors; this is a last-resort guard
                    output = {"status": "error", "error": f"Tool '{tool_name}' raised an exception: {exc}"}

        content = output if isinstance(output, str) else json.dumps(output)
        return {"role": "tool", "tool_call_id": tool_call.id, "content": content}

    def chat(self, user_message: str) -> str:
        """Send one user turn through the tool-use loop and return the final text answer."""
        self.memory.add_user_message(user_message)

        for _ in range(self.max_turns):
            response = self._call_llm()
            message = response.choices[0].message

            assistant_entry: Dict[str, Any] = {"role": "assistant", "content": message.content}
            if message.tool_calls:
                assistant_entry["tool_calls"] = [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {"name": tc.function.name, "arguments": tc.function.arguments},
                    }
                    for tc in message.tool_calls
                ]
            self.memory.add_message(assistant_entry)

            if not message.tool_calls:
                return message.content or ""

            for tool_call in message.tool_calls:
                self.memory.add_message(self._execute_tool_call(tool_call))

        return (
            f"Reached the maximum of {self.max_turns} tool-call turns without a final answer. "
            "Try a more specific question, or increase max_turns."
        )


agent = ResearchAgent(memory)
print("Agent ready (model:", MODEL_NAME, "via OpenRouter).")

Agent ready (model: openrouter/free via OpenRouter).


## 10. Demonstration

Shows the agent using all four tools together to answer multi-hop questions in a single session, satisfying the assignment's core requirement.

### Test Data Setup

Creates a sample `.txt` and `.pdf` file for the file reader tool to read during the demonstration.

In [11]:
# sample .txt
with open("sample_report.txt", "w") as f:
    f.write(
        "Acme Robotics Inc. -- Internal Report\n"
        "Founded: 2015\n"
        "Headquarters: Austin, Texas\n"
        "2023 Revenue: $42 million\n"
        "CEO: Jordan Lee\n"
        "Main product line: warehouse automation robots\n"
    )

# sample .pdf
from fpdf import FPDF

pdf = FPDF()
pdf.add_page()
pdf.set_font("Helvetica", size=12)
for line in [
    "Acme Robotics - Supplemental Filing",
    "Employees: 310",
    "Primary competitor: Bolt Dynamics",
]:
    pdf.cell(0, 10, text=line, new_x="LMARGIN", new_y="NEXT")
pdf.output("sample_report.pdf")

print("Created sample_report.txt and sample_report.pdf")

Created sample_report.txt and sample_report.pdf


### Example 1: File Reading and Memory Write

Demonstrates the file reader tool together with the memory-write capability: the agent reads a fact from a file and stores it for later recall.

In [12]:
answer1 = agent.chat(
    "Read the file 'sample_report.txt' and tell me which city Acme Robotics is "
    "headquartered in. Save that city as a fact called 'acme_hq_city' so we can "
    "reuse it later in this conversation."
)
print("AGENT:", answer1)

[HOOK] #5 read_file({'path': 'sample_report.txt', 'max_chars': 5000}) -> 3.2ms
[HOOK] #6 remember_fact({'key': 'acme_hq_city', 'value': 'Austin, Texas'}) -> 0.02ms
AGENT: Acme Robotics blends cutting‑edge robotics with logistics solutions.  
The internal report (file: **sample_report.txt**) lists its headquarters as **Austin, Texas**.

I have stored this location as the fact `acme_hq_city` for use later in the conversation.


### Example 2: Memory Recall and Web Search

Demonstrates multi-hop reasoning: the agent recalls the fact stored in Example 1 from memory, then uses it to perform a live web search, without the fact being repeated in the prompt.

In [13]:
answer2 = agent.chat(
    "Without me repeating it: recall the city where Acme Robotics is headquartered "
    "from memory, then search the web for the current weather in that city, and "
    "tell me both the city and the weather you found."
)
print("AGENT:", answer2)

[HOOK] #7 recall_fact({'key': 'acme_hq_city'}) -> 0.06ms
[HOOK] #8 web_search({'query': 'current weather in Austin, Texas', 'count': 5}) -> 15783.27ms
AGENT: The recall was successful: Acme Robotics is headquartered in **Austin, Texas**.

Unfortunately, the web search for the current weather timed out, so I don’t have a fresh weather reading. You can quickly check a weather service (e.g., weather.com) for “Austin, Texas” to get the latest conditions.


### Example 3: PDF Reading and Cross-Verification

Demonstrates the PDF reader combined with memory and web search: the agent reads a second file, then cross-checks the result against memory and a live search within a single multi-hop turn.

In [14]:
answer3 = agent.chat(
    "Read 'sample_report.pdf' and tell me who Acme Robotics' primary competitor is. "
    "Then, using what you already know from memory about Acme's headquarters city, "
    "search the web to check whether that competitor is also based in the same city."
)
print("AGENT:", answer3)

[HOOK] #9 read_file({'path': 'sample_report.pdf', 'max_chars': 5000}) -> 34.64ms
[HOOK] #10 web_search({'query': 'Bolt Dynamics headquarters city', 'count': 5}) -> 3025.43ms
AGENT: **Primary competitor (from `sample_report.pdf`):** Acme Robotics' primary competitor is **Bolt Dynamics**.

**Headquarters city (from memory):** Acme Robotics is headquartered in **Austin, Texas** (stored as `acme_hq_city`).

**Is Bolt Dynamics also in Austin?** Based on the web search, the Bolt Dynamics result (bolt-dynamics.com) lists its address as **18400 S Broadway St, Gardena, CA 90248** — i.e., **Gardena, California**, not Austin. So no, the competitor is **not** based in the same city as Acme Robotics.

(Note: the search also surfaced unrelated "Dynamics" companies like Boston Dynamics and BotLab Dynamics, which are not the same as Bolt Dynamics.)


## 11. Logs

Prints the full tool-call log recorded by the logging hook, confirming that every tool invocation above (search, file reads, memory reads and writes) was captured with timestamps and duration.

In [15]:
print(f"Total tool calls logged: {len(TOOL_CALL_LOG)}\n")
print(json.dumps(TOOL_CALL_LOG, indent=2))

Total tool calls logged: 10

[
  {
    "call_id": 1,
    "tool": "read_file",
    "inputs": {
      "path": "this_file_does_not_exist.txt",
      "max_chars": 5000
    },
    "start_time": "2026-07-17T01:35:23.277",
    "end_time": "2026-07-17T01:35:23.278",
    "duration_ms": 0.07,
    "status": "success",
    "result_preview": "{'status': 'error', 'path': 'this_file_does_not_exist.txt', 'error': \"File not found: 'this_file_does_not_exist.txt'\"}"
  },
  {
    "call_id": 2,
    "tool": "read_file",
    "inputs": {
      "path": "scratch_unsupported.md",
      "max_chars": 5000
    },
    "start_time": "2026-07-17T01:35:23.278",
    "end_time": "2026-07-17T01:35:23.278",
    "duration_ms": 0.28,
    "status": "success",
    "result_preview": "{'status': 'error', 'path': 'scratch_unsupported.md', 'error': \"Unsupported file type '.md'. Only .txt and .pdf are supported.\"}"
  },
  {
    "call_id": 3,
    "tool": "recall_fact",
    "inputs": {
      "key": "   "
    },
    "start_time": 

## 12. Memory State

Prints the final contents of the agent's fact store, confirming that facts written during the demonstration persisted for the rest of the session.

In [16]:
print(memory.facts_summary())

- acme_hq_city: Austin, Texas


## 13. Conclusion

This notebook implements a research agent that satisfies the Week 4 requirements: a web search tool backed by SerpApi, session memory with explicit fact storage and recall, a logging hook that records every tool call, a file reader supporting `.txt` and `.pdf`, and a single agent that combines all of these to answer multi-hop questions.

The agent runs on OpenRouter using OpenAI-compatible tool calling. The four tools, the memory store, and the logging hook are implemented as independent, reusable components. The Demonstration section shows the agent chaining file reading, memory, and web search across three turns to answer questions that require information from more than one source.

The web search integration was originally built on the Brave Search API and later migrated to SerpApi, with the tool's interface and output format preserved across that change. The notebook has also been reviewed for consistent error handling, structured tool outputs, and retry behavior on transient API failures.